# Deliverable 3 - INF201 

Submission deadline: 13th Nov 2026

### Hand-in information:
* This is a group submission, so one person submits on behalf of the pair/group
* Submit Python scripts (.py), a Jupyter notebook (ipynb) <i>and</i> a pdf of the same notebook
    * See Overview and objectives for a list of what your notebook should contain
* Include a link to your GitHub repository and a log of the commits
    * To generate the log, paste this into the terminal after your last commit:<br>
    `git log --graph --pretty=format:"%h - %an, %ar : %s" > git_log.txt`
    * The repository should contain a `requirements.txt` with information on your virtual environment
    * Note: Remember to create a new virtual environment for each deliverable
* Include a discussion of GenAI use, for example a summary of the chat log as you have
done earlier
    * This can be a separate file or be included in the Jupyter notebook
* Make a short (5 mins) video discussing your code and results
    * Note that you both have to contribute to the video
    * Either by recording it together
    * Or recording parts separately and editing it together
* We would appriciate it if use English for this assignment, both in the writing and in the video

## Overview and objectives

In this assignment you will continue working on the Character class in `game_simulator_v1.py` that we created in lecture_08 (check out the Week 44 module), and expand this to create a mini-RPG in the form of a battle simulator. To do this you will update and extend the existing class and methods, introduce new classes allowing us to use dice and weapons, and create a combat manager class.

The rules for the game is as follows:
* The game you build should allow two characters to battle until one of them wins. 
* The characters can use weapons to attack, where the damage dealt is determined by a dice roll.
* The characters can have different specialisations (warrior or mage). 
* A character's HP (hit points, or health) must be between 0 and the max given for this specific character.
* The character dies when it reaches 0 HP.

Learning Objectives:
* Enforce data encapsulation using private attributes and properties.
* Implement custom classes to model discrete concepts (`Dice`, `Weapon`, `Battle`).¨
* Create subclasses inheriting from the `Character` class.
* Apply object composition by equipping characters with weapons and dice.

To organise your project, we recommend the following struckture:
* Separate modules (Python scripts) for each of the classes created
* A Jupyter Notebook containing the following:
    * A presentation of each class created (showing attributes and methods objects of the class has)
    * Simulated battles between different types of characters, with and without weapons

## Task 1: Update the Character class

When we created the Character class in the lecture we first included an `attack()` method, and afterwards indicated that we wanted hit points to be a protected feature `_hp`. This lead to the `Character.attack()` method changing the target's protected attribute. Modifying another object's private attribute is bad form! Your first task is to fix this.

* Implement a `take_damage()` method to be used when the `Character` object is injured. This method should only affect the characters *own* HP.
* Implement a `heal()` method to be used when the `Character` heals (in other words increase their hit points)
* Change the existing `attack()` method so that calls the target's `take_damage()` method rather than directly modifying the target's `_hp`.
* Add an `is_alive` property to the `Character` class. This should be read-only, and return True if `_hp`$ > 0$ and False otherwise.

In [65]:
from character import Character, Warrior, Mage
from dice import Dice
from items import Weapon
from battle import Battle

hero = Character("Jacob", hp_max=50, damage=8)
hero                        
print(hero)        
print("Attributes:", vars(hero))
print("Methods:", [m for m in dir(hero) if not m.startswith("_")])

Character: Jacob with 50 HP and 8 damage
Attributes: {'name': 'Jacob', '_hp': 50, 'hp_max': 50, 'damage': 8, 'weapon': None}
Methods: ['attack', 'damage', 'heal', 'hp', 'hp_max', 'introduce', 'is_alive', 'name', 'take_damage', 'weapon']


In [63]:
hero.take_damage(20)
print("hp after 20 damage is", hero.hp)    # 30

#heal() makes hp go up, but never above its max, as shown here:
hero.heal(10)
print("hp after healing 10:", hero.hp)   # 40
hero.heal(1000)
print("hp after healing 1000:", hero.hp) # 50 (capped at hp_max)

hp after 20 damage is 30
hp after healing 10: 40
hp after healing 1000: 50


In [64]:
enemy = Character("Jafrin", hp_max=40, damage=6)
hero.attack(enemy)
print("enemy hp is", enemy.hp)  # should be 40 - 8 is 32


enemy hp is 32


In [7]:
enemy.take_damage(100)                 # more than enough to kill
print("Enemy HP:", enemy.hp)           # 0
print("Enemy alive:", enemy.is_alive)  # False
print("Hero alive:", hero.is_alive)    # True

Enemy HP: 0
Enemy alive: False
Hero alive: True


## Task 2: Introducing randomness with a Dice class

Currently, characters deal fixed damage. To introduce variable damage, create a separate `Dice` class.

* Create a `Dice` class that can create any n-sided Dice (n>=2), where the default number of sides is 6.
* Add a `roll()` method to roll the dice, using `random.randint(1, self.sides)`.
* Add a `__repr__` method that returns string representations like `Dice(sides=6)`.
* Implement a `__call__` method that allow you to roll the dice created by `d20 = Dice(sides=20)` by calling `d20.roll()`.
    * See the [Python documentation of `__call__`](https://docs.python.org/3/reference/datamodel.html#object.__call__) or [this example](https://www.w3schools.com/python/python_magic_call.asp)

In [69]:
#the default is 6 sides, but it can be changed if n>2
d6 = Dice()
d20 = Dice(sides=20)

print("d6 sides:", d6.sides) 
print("d20 sides:", d20.sides)   


d6 sides: 6
d20 sides: 20


In [71]:
#roll uses a random integer between 1 and "dice".sides
rolls = [d6.roll() for _ in range(12)]
print("12 rolls of d6:", rolls)
print("All results between 1 and 6:", all(1 <= r <= 6 for r in rolls))

12 rolls of d6: [1, 5, 2, 1, 1, 5, 2, 3, 4, 3, 2, 2]
All results between 1 and 6: True


In [ ]:
#calling makes the dice object behave as a function and calling it should give the same result as just using it
print("d20() gives:     ", d20())
print("d20.roll() gives: ", d20.roll())

d20() gives:      11
d20.roll() gives:  17


## Task 3: Create items

Create a `Weapon` class in `items.py` that utilizes your `Dice` class for damage calculations. We call the module `items.py` rather than `weapons.py` so that it can be expanded to include other items your characters can hold.

* Define the class `Weapon` with the attributes `name` and `damage_dice: Dice`.
* Include a `roll_damage()` method that calls `self.damage_dice.roll()`
* Update `Character.__init__` to accept an optional weapon.
* If a character has an equipped weapon, `attack()` should roll damage from the weapon instead of using default base damage.

In [72]:
#a weapon holds bothe the name of it and the dice being used for damage
sword = Weapon("Sword", Dice(8))

print("Name:", sword.name)
print("Damage dice:", sword.damage_dice)
print("Attributes:", vars(sword))

Name: Sword
Damage dice: Dice(sides=8)
Attributes: {'name': 'Sword', 'damage_dice': Dice(sides=8)}


In [ ]:
#the character init accapts an optional weapon, but defaults to no weapon
armed = Character("Vilja", weapon=sword)
unarmed = Character("Max", damage=7)

print("Armed weapon:", armed.weapon.name) 
print("Unarmed weapon:", unarmed.weapon)

Armed weapon: Sword
Unarmed weapon: None


In [73]:
#a weapon uses a dice which makes the mage vary, but the damage is constant unarmed
target = Character("Dummy", hp_max=10_000)

weapon_damages = []
for _ in range(8):
    before = target.hp
    armed.attack(target)
    weapon_damages.append(before - target.hp)
print("Damage with sword:", weapon_damages)
base_damages = []
for _ in range(8):
    before = target.hp
    unarmed.attack(target)
    base_damages.append(before - target.hp)
print("Damage without weapon:", base_damages)

Damage with sword: [2, 4, 1, 1, 5, 1, 7, 2]
Damage without weapon: [7, 7, 7, 7, 7, 7, 7, 7]


## Task 4: Character specialization

Extend the character module to include two specialised subclasses of `Character`.

1. `Warrior` subclass (defensive specialist)
    * New attribute: `armor: int`, with a default value of 5
    * Override the `Character.take_damage()` method by reducing the incoming damage by armor before applying it.
2. `Mage` subclass (resource-based attacker)
    * New attributes: `mana: int` and `mana_max: int`, both with default value 30
    * Magic attacks consume mana, so the mana available determines whether the mage can use magic.
    * Override the `Character.attack()` method:
        * If the mage has at least 10 mana left, spend 10 mana to deal a magical spell attack dealing double damage. This magic attack does not use any weapons, even if the character has one equipped.
        * If the mage has less than 10 mana left, fall back to the base attack, and restore 5 mana. This base attack should use whichever weapon the character has equipped.

In [ ]:
#the warrior has a new attribute, armor, it defaults to 5
warrior = Warrior("Jacob")
print("Attributes:", vars(warrior))
print("Warrior is a Character:", isinstance(warrior, Character)) #shows inheritence

#take_damage is overriden and armor blocks 5 of incoming damage
warrior.take_damage(12)
print("HP after 12 damage:", warrior.hp)
warrior.take_damage(3)      # will be fully blocked
print("HP after 3 damage:", warrior.hp)  

Attributes: {'name': 'Jacob', '_hp': 100, 'hp_max': 100, 'damage': 10, 'weapon': None, 'armor': 5}
Warrior is a Character: True
HP after 12 damage: 93
HP after 3 damage: 93


In [ ]:
#mage has mana and max_mana as new attributes
mage = Mage("Jafrin")
print("Attributes:", vars(mage))

dummy = Character("Dummy", hp_max=10_000)
before = dummy.hp
#the attack will use 10 mana and do double damage (20)
mage.attack(dummy)
print("Spell damage:", before - dummy.hp) 
print("Mana after spell:", mage.mana) 

Attributes: {'name': 'Jafrin', '_hp': 100, 'hp_max': 100, 'damage': 10, 'weapon': None, 'mana': 30, 'mana_max': 30}
Spell damage: 20
Mana after spell: 20


In [ ]:
#magic attacks wont use weapon if mana is available
armed_mage = Mage("Jafrin", weapon=Weapon("Big_stick", Dice(12)))
target = Character("Dummy", hp_max=10_000)

before = target.hp
armed_mage.attack(target)
print("Damage with stick equipped:", before - target.hp)  #damage is still 20

Damage with stick equipped: 20


In [81]:
#when under 10 mana, use weapo instead and restore 5 mana
mage = Mage("Jafrin", weapon=Weapon("staff", Dice(12)))
target = Character("Dummy", hp_max=10_000)

for i in range(4):
    hp_before, mana_before = target.hp, mage.mana
    mage.attack(target)
    print(f"Attack {i+1}: mana {mana_before} to {mage.mana}, "
          f"damage was {hp_before - target.hp}")

Attack 1: mana 30 to 20, damage was 20
Attack 2: mana 20 to 10, damage was 20
Attack 3: mana 10 to 0, damage was 20
Attack 4: mana 0 to 5, damage was 5


## Task 5: Running battles

Create a `Battle` class to manage turn-based combat between two `Character` objects.

* Initialize `Battle` with `hero: Character` and `enemy: Character`.
* Implement a `play_round()` method where `hero` attacks `enemy`, and if `enemy` is still alive, `enemy` attacks `hero`.
* Implement a `simulate_battle()` method with a `while` loop that runs rounds until one character is no longer alive.

In [82]:
#the battle class holds both fighters so it can use them
battle = Battle(Warrior("Jacob", weapon=Weapon("Axe", Dice(10))), Mage("Jafrin"))

#this is one round. hero starts by attacking and enemy attacks after if its still alive
battle.play_round()
print("Hero HP:", battle.hero.hp)
print("Enemy HP:", battle.enemy.hp)
print("Enemy mana:", battle.enemy.mana)

Hero HP: 85
Enemy HP: 97
Enemy mana: 20


In [ ]:
#the enemy doesnt hit back if its dead
quick = Battle(Character("Jacob", damage=10), Character("Sucker", hp_max=5))
quick.play_round()
print("Enemy alive?", quick.enemy.is_alive) 
print("Hero HP:", quick.hero.hp)    #I wont lose any hp

Enemy alive? False
Hero HP: 100


In [85]:
#simulate battle plays turns until one is dead
battle2 = Battle(Warrior("Jacob", weapon=Weapon("Spear", Dice(10))),
                 Mage("Jafrin", weapon=Weapon("Wand", Dice(6))))
battle2.simulate_battle()

if battle2.hero.is_alive:
    print("Winner:", battle2.hero.name)
else:
    print("Winner:", battle2.enemy.name)

Winner: Jafrin


here are some more battles as stated as needed between different weapons and characters

In [ ]:
#Warrior vs a mage, both have weapons
b1 = Battle(Warrior("Jacob", weapon=Weapon("Axe", Dice(10))),
            Mage("Jafrin", weapon=Weapon("Staff", Dice(6))))
b1.simulate_battle()
if b1.hero.is_alive:
    print("With weapons winner is", b1.hero.name)
else:
    print("With weapons winner is", b1.enemy.name)

With weapons winner is Jafrin


In [60]:
#plain characters with no weapons and fixed damage
b2 = Battle(Character("Vilja", damage=8), Character("Guro", damage=12))
b2.simulate_battle()
if b2.hero.is_alive:
    print("Without weapons winner is", b2.hero.name)
else:
    print("Without weapons winner is", b2.enemy.name)

Without weapons winner is Guro


In [62]:
#Mage vs warrior with no weapons
b3 = Battle(Mage("Jafrin"), Warrior("Jacob"))
b3.simulate_battle()
if b3.hero.is_alive:
    print("mage vs warrior winner is", b3.hero.name)
else:
    print("mage vs warrior winner is", b3.enemy.name)

mage vs warrior winner is Jafrin
